# Llama3 Base Model Evaluation

## Import Libraries

In [ ]:
!pip install -q --upgrade bitsandbytes

In [ ]:
!wget -q https://raw.githubusercontent.com/KumudithaSilva/llama3-domain-adaptation/feature-base-model/evaluator.py -O evaluator.py

In [ ]:
import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset, Dataset, DatasetDict
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
from peft import LoraConfig, PeftConfig
from datetime import  datetime
from evaluator import evaluate

## Load Dataset From HuggingFace

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"

PROJECT_NAME = "stream_price"

RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"

DATA_USER = "KumudithaSilva"
DATASET_NAME = f"{DATA_USER}/stream_items_prompt_lite"

In [ ]:
hf_token = userdata.get('HUGGING_KEY')
login(hf_token)

In [ ]:
dataset = load_dataset(DATASET_NAME)

train = dataset['train'].remove_columns(['id'])
val = dataset['validation'].remove_columns(['id'])
test = dataset['test'].remove_columns(['id'])

In [ ]:
train[0]

## Load Llama Model

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
    )

## Llama Model

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    )

## Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

## Memory Footprint

In [ ]:
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:.1f} GB")

## Model Prediction

In [ ]:
def model_predict(item):
    inputs = tokenizer(item["prompt"],return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = base_model.generate(**inputs, max_new_tokens=8)

    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]

    return tokenizer.decode(generated_ids)

In [ ]:
test[0]

## Evaluation

In [ ]:
evaluate(model_predict, test)